# Fase 4 — Regresión lineal y validación cruzada

**TP1 — Aprendizaje Automático (72.75) — ITBA** · Consignas **2.2** y **2.3**

Pasos **7 (Modelado)** y **9 (Evaluación en Dev)** del pipeline de la Clase 3.

Implementamos el k-fold sobre el train, entrenamos la regresión lineal dentro de la
validación cruzada y reportamos el RMSE de train y de validación.

**El test no se abre en este notebook.**

In [1]:
# ---------------------------------------------------------------------------
# Configuracion del entorno
# ---------------------------------------------------------------------------
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.config import TARGET, RANDOM_SEED, N_SPLITS
from src.data import cargar_splits
from src.preprocesamiento import separar_X_y

# Funciones propias, documentadas en src/modelado.py
from src.modelado import (
    crear_kfold,          # arma el KFold con la semilla del proyecto
    crear_modelo_lineal,  # Pipeline: preprocesamiento + LinearRegression
    evaluar_cv,           # corre la validacion cruzada y devuelve RMSE por fold
    tabla_coeficientes,   # extrae los coeficientes con el nombre de cada feature
)

pd.set_option("display.width", 120)

# Solo el train. El test queda cerrado hasta la Fase 6.
train, _ = cargar_splits()
X_train, y_train = separar_X_y(train)

print(f"Train: {X_train.shape[0]} filas | {X_train.shape[1]} variables de entrada")

Train: 1069 filas | 6 variables de entrada


---

# 1. Esquema de validación cruzada *(consigna 2.2)*

La consigna pide implementar k-fold **usando únicamente el conjunto de entrenamiento**.

**Cómo funciona.** El train se parte en 5 bloques. En cada iteración, 4 bloques entrenan el
modelo y 1 lo valida. Se repite 5 veces, rotando cuál es el bloque de validación, de modo
que **cada observación se valida exactamente una vez**.

**Por qué k-fold y no un dev set fijo.** La Clase 2 (slide 83) propone 60-20-20 con un dev
fijo. Preferimos k-fold porque, como señala la Clase 3 (slide 15), *"cada observación se
utiliza una vez para validación y en el resto de los folds para entrenamiento"*, y porque
*"reducimos nuestra dependencia de un único validation split, que puede ser afortunado o
desafortunado"*. Con 1069 filas, reservar 214 sólo para elegir modelo sería desaprovecharlas.

**Por qué k = 5.** La Clase 2 (slide 85) indica que *"k = 5 o 10 son las más usadas y son
suficientes"*. Con k = 5 cada fold de validación tiene ~214 observaciones.

In [2]:
# KFold con shuffle y semilla fija: el reparto de filas entre folds es siempre el mismo.
kf = crear_kfold()

print(f"Folds: {kf.get_n_splits()}")
for i, (idx_ajuste, idx_val) in enumerate(kf.split(X_train), start=1):
    print(f"  Fold {i}: entrena con {len(idx_ajuste)} filas, valida con {len(idx_val)}")

Folds: 5
  Fold 1: entrena con 855 filas, valida con 214
  Fold 2: entrena con 855 filas, valida con 214
  Fold 3: entrena con 855 filas, valida con 214
  Fold 4: entrena con 855 filas, valida con 214
  Fold 5: entrena con 856 filas, valida con 213


---

# 2. El modelo

La regresión lineal busca los coeficientes que minimizan la suma de los errores al cuadrado
(Clase 2, slide 50). Va encadenada con el preprocesamiento de la Fase 3 en un único
`Pipeline`.

In [3]:
# crear_modelo_lineal() devuelve un Pipeline de dos pasos:
#   1. preprocesamiento: one-hot para categoricas + z-score para numericas
#   2. regresion       : LinearRegression
modelo_lineal = crear_modelo_lineal()
modelo_lineal

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesamiento', ...), ('regresion', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feat

**Por qué el preprocesamiento va adentro del `Pipeline` y no antes.**

`cross_validate` reajusta **toda la cadena** dentro de cada fold: el escalador aprende la
media y el desvío del sub-conjunto de entrenamiento de ese fold, no de todo el train.

Si transformáramos los datos una sola vez antes del k-fold, cada fold de validación habría
participado en el cálculo de esos estadísticos y el error de validación saldría optimista.
Es la regla 2 del proyecto, garantizada por construcción.

---

# 3. Entrenamiento y evaluación *(consigna 2.3)*

La métrica es el **RMSE** (Clase 3, slide 87): la raíz del promedio de los errores al
cuadrado. Queda en las mismas unidades que el target, así que se lee directamente en dólares.

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Reportamos **train y validación**, como pide la consigna. Como métrica complementaria
agregamos el **R²** (Clase 3, slide 88), que indica qué proporción de la variabilidad del
costo explica el modelo.

In [4]:
# evaluar_cv entrena el modelo en cada fold y mide el error en las dos partes:
# en los datos con los que se entreno (train) y en los que se reservaron (validacion).
por_fold, resumen_lineal = evaluar_cv(modelo_lineal, X_train, y_train, "Regresion lineal", cv=kf)

print("RMSE y R2 en cada fold:\n")
por_fold.round(2)

RMSE y R2 en cada fold:



,rmse_train,rmse_val,r2_train,r2_val
fold,,,,
1,6139.56,5846.90,0.73,0.73
2,6104.44,6003.92,0.73,0.72
3,6073.79,6130.97,0.72,0.77
4,6051.68,6239.26,0.74,0.69
5,6010.12,6397.21,0.73,0.71


In [5]:
# Promedio sobre los 5 folds.
print(f"RMSE train      : {resumen_lineal['rmse_train']:>10,.2f}")
print(f"RMSE validacion : {resumen_lineal['rmse_val']:>10,.2f}")
print(f"Desvio del RMSE de validacion entre folds: {resumen_lineal['rmse_val_desvio']:,.2f}")
print(f"R2 validacion   : {resumen_lineal['r2_val']:>10.3f}")
print(f"Gap (val - train): {resumen_lineal['gap']:>9,.2f}")

RMSE train      :   6,075.92
RMSE validacion :   6,123.65
Desvio del RMSE de validacion entre folds: 211.65
R2 validacion   :      0.723
Gap (val - train):     47.73


---

# 4. ¿El modelo hace overfitting?

| | Valor |
|---|---|
| RMSE train | **\$6.075,92** |
| RMSE validación | **\$6.123,65** |
| Gap | **\$47,73** (0.8% del RMSE) |

La Clase 2 (slides 76–77) define el *generalization gap* como la diferencia entre el error de
validación y el de train, y señala que *"si el gap es cero, el modelo generaliza bien"*.

**El gap es de 48 dólares sobre un error de 6.124: prácticamente cero.** El modelo no está
memorizando ruido del entrenamiento.

Tiene sentido: son 8 parámetros ajustados con 1069 observaciones. Con esa relación, un
modelo lineal no tiene capacidad suficiente para memorizar. Si acaso, el riesgo es el
contrario —**underfitting**, que el modelo sea demasiado simple para el problema—, y es
justamente lo que la Fase 5 va a poner a prueba con la regresión polinómica.

Dicho eso, en términos absolutos el error es alto: **\$6.124 equivale al 47% del costo medio
del train (\$13.030)**, y el R² de validación es 0.72, o sea que queda un 28% de la
variabilidad sin explicar.

---

# 5. Qué aprendió el modelo

Entrenamos el modelo con **todo el train** para leer sus coeficientes. Las variables
numéricas están estandarizadas, así que su coeficiente indica cuánto cambia el costo predicho
al aumentar esa variable **en un desvío estándar**. Las columnas one-hot valen 0 o 1, así que
su coeficiente es la diferencia respecto de la categoría de referencia.

In [6]:
# Entrenamos con todo el train para inspeccionar los coeficientes.
modelo_lineal.fit(X_train, y_train)

coeficientes = tabla_coeficientes(modelo_lineal)
print(f"Intercepto: {modelo_lineal.named_steps['regresion'].intercept_:,.2f}\n")
coeficientes.round(2)

Intercepto: 8,947.95



,coeficiente,coef_abs
smoker_yes,23077.76,23077.76
age,3472.98,3472.98
bmi,1927.83,1927.83
region_southeast,-838.92,838.92
region_southwest,-659.14,659.14
children,636.50,636.50
region_northwest,-391.76,391.76
sex_male,-101.54,101.54


**Lo que dicen los coeficientes:**

- **`smoker_yes` = +23.078.** Un fumador cuesta unos \$23.000 más por año que un no fumador
  con las mismas características. Es cinco veces más grande que el siguiente coeficiente, y
  confirma numéricamente lo que el EDA mostró en gráficos.
- **`age` = +3.473.** Cada desvío estándar de edad (unos 14 años) suma ~\$3.500.
- **`bmi` = +1.928** por desvío estándar (unos 6 puntos de BMI).
- **`sex_male` = −102.** Prácticamente cero: el sexo casi no influye, como ya anticipaba el
  EDA (medias de \$12.305 vs \$13.714).
- **Las regiones** aportan entre −392 y −839 respecto de `northeast`: diferencias chicas.

Este orden es coherente con los filtros de la Fase 3, y adelanta lo que va a hacer la
regularización L1 en la Fase 5: los primeros candidatos a que su coeficiente vaya a cero son
`sex_male` y las regiones.

---

## Conclusiones de la Fase 4

| Decisión / resultado | Valor |
|---|---|
| Esquema de validación | k-fold con **k = 5**, sólo sobre el train |
| Modelo | `Pipeline`: preprocesamiento + `LinearRegression` |
| **RMSE train** | **\$6.075,92** |
| **RMSE validación** | **\$6.123,65** |
| Desvío del RMSE entre folds | \$211,65 |
| Gap (val − train) | \$47,73 → sin overfitting |
| R² validación | 0.72 |

**Lectura:** el modelo lineal no hace overfitting, pero deja un error alto en términos
absolutos (47% del costo medio). Como el gap es casi nulo, el margen de mejora no está en
regularizar sino en **darle más capacidad al modelo**.

**Siguiente:** Fase 5 — regresión polinómica y regularización L1 (consignas 3.1, 3.2, 3.3 y 4).